### Import Libraries

In [1]:
# Ignore warnings
import warnings
warnings.filterwarnings("ignore")
import random
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from pathlib import Path
from torch.utils.data import DataLoader
from dataloader import ACDCDataset
from src.org_unet import UNet
from src.residual_unet import ResidualUNet
from src.attention_unet import AttentionUNetV3
from src.feature_pyramid_unet import FeaturePyramidUNet
from src.feedback_resunet import FeedbackResUNet
from src.transUnet import TransformerUNet
from src import SegmentationMetrics, evaluate_dice_score
import matplotlib.pyplot as plt
%matplotlib inline

INFO:albumentations.check_version:A new version of Albumentations is available: 2.0.5 (you have 1.4.7). Upgrade using: pip install --upgrade albumentations


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = torch.device('cpu')
print(f'Using device {device}')

Using device cuda


### Data Loading

In [3]:
root_dir = r'../data/ACDC/img_slices_org_v2/' #img_slices_with_ratios_v4
# training_dataset = ACDCDataset(root_dir=root_dir, dataset='training')
# train_dataloader = DataLoader(training_dataset, batch_size=8, shuffle=True)
testing_dataset = ACDCDataset(root_dir=root_dir, dataset='testing')
test_dataloader = DataLoader(testing_dataset, batch_size=32, shuffle=True)

# torch.manual_seed(18)

Loaded 260 testing images


### Model Inferencing

In [4]:
def load_model(checkpoint_path, model_class):
    """
    Load a model from a given checkpoint.

    Parameters:
        checkpoint_path (str): Path to the model checkpoint.
        model_class (torch.nn.Module): Class of the model architecture.

    Returns:
        torch.nn.Module: Loaded model.
    """
    model = model_class()
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint)
    model.eval()
    return model

In [5]:
def visualize_predictions(dataloader, models, model_names, device, num_samples=5, classes=None):
    """
    Visualizes predictions of multiple models alongside the original image and ground truth mask.

    Parameters:
        dataloader (torch.utils.data.DataLoader): The DataLoader providing the data.
        models (list of torch.nn.Module): List of trained models for predictions.
        model_names (list of str): Names of the models for display.
        device (torch.device): The device to run the models (e.g., 'cuda' or 'cpu').
        num_samples (int): Number of samples to visualize from the dataloader.
        classes (list): Optional list of class names for displaying masks.
    """
    # for model in models:
    #     model.eval()  # Set all models to evaluation mode

    with torch.no_grad():  # Disable gradient calculation
        for i, sample in enumerate(dataloader):
            if i >= num_samples:
                break

            images, masks, idx = sample['image'], sample['masks_all'], sample['idx']
            # print(f"original images: shape {images.shape} | dtype {images.dtype}")
            images = images.permute(0, 3, 1, 2)  # Ensure proper format for PyTorch models
            # print(f"permuted images: shape {images.shape} | dtype {images.dtype}")
            images = images.to(device=device, dtype=torch.float32, memory_format=torch.channels_last)
            # print(f"device transformed images: shape {images.shape} | dtype {images.dtype}")
            masks = masks.to(device=device, dtype=torch.long)

            batch_size = images.size(0)
            for j in range(batch_size):
                if num_samples <= 0:
                    break

                num_samples -= 1

                # Prepare visualization grid
                fig, axes = plt.subplots(1, len(models) + 2, figsize=(3 * (len(models) + 2), 8))

                # Original image
                image = images[j].cpu().permute(1, 2, 0).numpy()  # Convert to HWC format
                axes[0].imshow(image, cmap="gray")
                axes[0].set_title(f"Original Image - {idx[j]}")
                axes[0].axis("off")

                # Ground truth
                ground_truth = masks[j].cpu().numpy()
                axes[1].imshow(ground_truth, cmap="gray", interpolation="none")
                axes[1].set_title("Ground Truth Mask")
                axes[1].axis("off")

                # Predictions from each model
                for k, (model, name) in enumerate(zip(models, model_names)):
                    # Move the input image to the same device as the model
                    input_image = images[j].unsqueeze(0)  # Add batch dimension
                    # print(f"input_image: shape {input_image.shape} | dtype {input_image.dtype}")
                    input_image = input_image.to(device)
                    # print(f"device transformed input_image: shape {input_image.shape} | dtype {input_image.dtype}")

                    model = model.to(device)

                    prediction = model(input_image)  # Get model prediction
                    prediction = torch.argmax(prediction, dim=1).squeeze(0).cpu().numpy()
                    # print(f"prediction: shape {prediction.shape} | dtype {prediction.dtype}")

                    axes[k + 2].imshow(prediction, cmap="gray", interpolation="none")
                    axes[k + 2].set_title(f"{name}")
                    axes[k + 2].axis("off")

                plt.tight_layout()
                plt.show()

In [6]:
# Load models
model_paths = [
    Path(f'./models/checkpoints/org_unet_img_slices_with_ratios_v4_20250101083911-handsome-moose-917/checkpoint_epoch{20}.pth'),
    Path(f'./models/checkpoints/res_unet_img_slices_with_ratios_v4_20241230195221-charming-skink-261/checkpoint_epoch{27}.pth'),
    Path(f'./models/checkpoints/attention_unet_img_slices_with_ratios_v4_20241226011819-fun-robin-567/checkpoint_epoch{10}.pth'),
    Path(f'./models/checkpoints/feat_pyramid_unet_img_slices_with_ratios_v4_20241224021156-rumbling-dog-720/checkpoint_epoch{23}.pth'),
    Path(f'./models/checkpoints/feedback_resunet_img_slices_with_ratios_v4_20241222225555-suave-whale-816/checkpoint_epoch{19}.pth'),
    Path(f'./models/checkpoints/transformer_unet_img_slices_with_ratios_v4_20241221213710-clumsy-tern-738/checkpoint_epoch{15}.pth')
]

model_classes = [
    UNet,
    ResidualUNet,
    AttentionUNetV3,
    FeaturePyramidUNet,
    FeedbackResUNet,
    TransformerUNet
]

models = [load_model(path, model_class) for path, model_class in zip(model_paths, model_classes)]

model_names = [
    "Original U-Net",
    "ResU-Net",
    "Attention U-Net",
    "Feature Pyramid U-Net",
    "Feedback ResU-Net",
    "Transformer U-Net"
]

In [7]:
# Visualize predictions
# visualize_predictions(test_dataloader, models, model_names, device=device, num_samples=5)

## Evaluation Metrics

In [7]:
def evaluate_model_on_loader(model, dataloader, device, num_classes=4, class_names=None):
    model.eval()
    all_preds = []
    all_gts = []

    with torch.no_grad():
        for sample in tqdm(dataloader, desc="Evaluating"):
            # Extract data from the sample
            images, true_masks = sample['image'], sample['masks_all']
            images = images.permute(0, 3, 1, 2)

            images = images.to(device=device, dtype=torch.float32, memory_format=torch.channels_last)
            true_masks = true_masks.to(device=device, dtype=torch.long)
            true_masks = F.one_hot(true_masks, model.n_classes).permute(0, 3, 1, 2).float()

            model = model.to(device)
            outputs = model(images)  # Output shape: (B, C, H, W)
            # preds = F.one_hot(outputs.argmax(dim=1), model.n_classes).permute(0, 3, 1, 2).float()
            preds = torch.argmax(outputs, dim=1)  # (B, H, W)
            # masks = true_masks.squeeze(1) if true_masks.ndim == 4 else true_masks  # Ensure shape: (B, H, W)

            # One-hot GT → Class index mask
            if true_masks.ndim == 4 and true_masks.shape[1] > 1:
                true_masks = torch.argmax(true_masks, dim=1)
            else:
                true_masks = true_masks.squeeze(1) if true_masks.ndim == 4 else true_masks

            all_preds.append(preds)
            all_gts.append(true_masks)

    all_preds = torch.cat(all_preds, dim=0)  # shape: (N, H, W)
    all_gts = torch.cat(all_gts, dim=0)

    metrics = SegmentationMetrics(all_preds, all_gts, num_classes=num_classes)

    avg_dice_scores_batch = evaluate_dice_score(model, dataloader, device=device)
    print(f"BKG: {avg_dice_scores_batch[0]:.4f}\tLV: {avg_dice_scores_batch[1]:.4f}\tRV: {avg_dice_scores_batch[2]:.4f}\tMYO: {avg_dice_scores_batch[3]:.4f}\tOVR: {round(torch.tensor(avg_dice_scores_batch).mean().item(), 4)}")
    print("dc_bkg", round(avg_dice_scores_batch[0].item(), 4))
    print("dc_lv", round(avg_dice_scores_batch[1].item(), 4))
    print("dc_rv", round(avg_dice_scores_batch[2].item(), 4))
    print("dc_myo", round(avg_dice_scores_batch[3].item(), 4))
    print("dice_score", round(torch.tensor(avg_dice_scores_batch).mean().item(), 4))

    return metrics.evaluate_per_class(class_names=class_names)

# Define class names: index 0 is background
class_names = ["Background", "LV", "RV", "MYO"]

In [8]:
# Original U-Net

results = evaluate_model_on_loader(models[0], test_dataloader, device, num_classes=4, class_names=class_names)

# Print Results
for cls, metrics in results.items():
    print(f"\n{cls}:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

Evaluating: 100%|██████████| 9/9 [00:15<00:00,  1.69s/it]


BKG: 0.9960	LV: 0.9411	RV: 0.8911	MYO: 0.8511	OVR: 0.9198
dc_bkg 0.996
dc_lv 0.9411
dc_rv 0.8911
dc_myo 0.8511
dice_score 0.9198

LV:
  Dice Score: 0.8731
  Jaccard Index (IoU): 0.8279
  Hausdorff Distance: 5.2168

RV:
  Dice Score: 0.7556
  Jaccard Index (IoU): 0.6968
  Hausdorff Distance: 31.1408

MYO:
  Dice Score: 0.7848
  Jaccard Index (IoU): 0.6928
  Hausdorff Distance: 7.2091


In [9]:
# ResU-Net

results = evaluate_model_on_loader(models[1], test_dataloader, device, num_classes=4, class_names=class_names)

# Print Results
for cls, metrics in results.items():
    print(f"\n{cls}:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

Evaluating: 100%|██████████| 9/9 [00:20<00:00,  2.25s/it]


BKG: 0.9950	LV: 0.9219	RV: 0.8449	MYO: 0.8204	OVR: 0.8956
dc_bkg 0.995
dc_lv 0.9219
dc_rv 0.8449
dc_myo 0.8204
dice_score 0.8956

LV:
  Dice Score: 0.8274
  Jaccard Index (IoU): 0.7771
  Hausdorff Distance: 5.0398

RV:
  Dice Score: 0.7065
  Jaccard Index (IoU): 0.6339
  Hausdorff Distance: 30.8211

MYO:
  Dice Score: 0.7393
  Jaccard Index (IoU): 0.6373
  Hausdorff Distance: 6.5437


In [10]:
# Attention U-Net

results = evaluate_model_on_loader(models[2], test_dataloader, device, num_classes=4, class_names=class_names)

# Print Results
for cls, metrics in results.items():
    print(f"\n{cls}:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

Evaluating: 100%|██████████| 9/9 [00:23<00:00,  2.65s/it]


BKG: 0.9956	LV: 0.9374	RV: 0.8810	MYO: 0.8403	OVR: 0.9136
dc_bkg 0.9956
dc_lv 0.9374
dc_rv 0.881
dc_myo 0.8403
dice_score 0.9136

LV:
  Dice Score: 0.8770
  Jaccard Index (IoU): 0.8284
  Hausdorff Distance: 3.7282

RV:
  Dice Score: 0.7342
  Jaccard Index (IoU): 0.6724
  Hausdorff Distance: 31.3621

MYO:
  Dice Score: 0.7762
  Jaccard Index (IoU): 0.6821
  Hausdorff Distance: 4.9347


In [11]:
# Feature Pyramid U-Net

results = evaluate_model_on_loader(models[3], test_dataloader, device, num_classes=4, class_names=class_names)

# Print Results
for cls, metrics in results.items():
    print(f"\n{cls}:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

Evaluating: 100%|██████████| 9/9 [00:18<00:00,  2.01s/it]


BKG: 0.9959	LV: 0.9445	RV: 0.8883	MYO: 0.8513	OVR: 0.92
dc_bkg 0.9959
dc_lv 0.9445
dc_rv 0.8883
dc_myo 0.8513
dice_score 0.92

LV:
  Dice Score: 0.8851
  Jaccard Index (IoU): 0.8368
  Hausdorff Distance: 4.8663

RV:
  Dice Score: 0.7512
  Jaccard Index (IoU): 0.6901
  Hausdorff Distance: 31.7252

MYO:
  Dice Score: 0.7913
  Jaccard Index (IoU): 0.6976
  Hausdorff Distance: 6.8218


In [12]:
# Feedback ResU-Net

results = evaluate_model_on_loader(models[4], test_dataloader, device, num_classes=4, class_names=class_names)

# Print Results
for cls, metrics in results.items():
    print(f"\n{cls}:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

Evaluating: 100%|██████████| 9/9 [01:15<00:00,  8.36s/it]


BKG: 0.9960	LV: 0.9389	RV: 0.8896	MYO: 0.8455	OVR: 0.9175
dc_bkg 0.996
dc_lv 0.9389
dc_rv 0.8896
dc_myo 0.8455
dice_score 0.9175

LV:
  Dice Score: 0.8839
  Jaccard Index (IoU): 0.8342
  Hausdorff Distance: 4.5408

RV:
  Dice Score: 0.7540
  Jaccard Index (IoU): 0.6917
  Hausdorff Distance: 29.7692

MYO:
  Dice Score: 0.7971
  Jaccard Index (IoU): 0.7015
  Hausdorff Distance: 5.6019


In [13]:
# Transformer U-Net

results = evaluate_model_on_loader(models[5], test_dataloader, device, num_classes=4, class_names=class_names)

# Print Results
for cls, metrics in results.items():
    print(f"\n{cls}:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

Evaluating: 100%|██████████| 9/9 [00:13<00:00,  1.46s/it]


BKG: 0.9931	LV: 0.8679	RV: 0.8013	MYO: 0.7191	OVR: 0.8454
dc_bkg 0.9931
dc_lv 0.8679
dc_rv 0.8013
dc_myo 0.7191
dice_score 0.8454

LV:
  Dice Score: 0.7525
  Jaccard Index (IoU): 0.6814
  Hausdorff Distance: 7.2436

RV:
  Dice Score: 0.5770
  Jaccard Index (IoU): 0.5048
  Hausdorff Distance: 32.3337

MYO:
  Dice Score: 0.6250
  Jaccard Index (IoU): 0.5081
  Hausdorff Distance: 9.6930
